In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import mean_squared_error, r2_score
from warnings import filterwarnings
filterwarnings('ignore')

In [2]:
hit = pd.read_csv("hitters.csv")
df = hit.copy()
df = df.dropna()
dms = pd.get_dummies(df[["League", "Division", "NewLeague"]])
dms = dms.astype(int)
y = df["Salary"]
X_ = df.drop(["Salary", "League", "Division", "NewLeague"], axis = 1).astype("float64")
X = pd.concat([X_, dms[["League_N", "Division_W", "NewLeague_N"]]], axis = 1)
X_train, X_test, y_train, y_test = train_test_split(X, 
                                                    y,
                                                    test_size = 0.25,
                                                    random_state = 42)

In [3]:
X.head()

,AtBat,Hits,HmRun,Runs,RBI,Walks,Years,CAtBat,CHits,CHmRun,CRuns,CRBI,CWalks,PutOuts,Assists,Errors,League_N,Division_W,NewLeague_N
1,315.0,81.0,7.0,24.0,38.0,39.0,14.0,3449.0,835.0,69.0,321.0,414.0,375.0,632.0,43.0,10.0,1,1,1
2,479.0,130.0,18.0,66.0,72.0,76.0,3.0,1624.0,457.0,63.0,224.0,266.0,263.0,880.0,82.0,14.0,0,1,0
3,496.0,141.0,20.0,65.0,78.0,37.0,11.0,5628.0,1575.0,225.0,828.0,838.0,354.0,200.0,11.0,3.0,1,0,1
4,321.0,87.0,10.0,39.0,42.0,30.0,2.0,396.0,101.0,12.0,48.0,46.0,33.0,805.0,40.0,4.0,1,0,1
5,594.0,169.0,4.0,74.0,51.0,35.0,11.0,4408.0,1133.0,19.0,501.0,336.0,194.0,282.0,421.0,25.0,0,1,0


*Aşağıda bir dönüştürme işlemi yapacağız. Aslında dönüştürme işlemini bütün makine öğrenmesi algoritmaları sever. Çünkü dönüştürme işlemi; Değişkenlerin göreceli etkilerinin belirli bir standart altında olmasını sağlar ve algoritmaların zorlanmasını engeller.*

*Bu durum YSA'da çok daha hassastır. Çok fazla katman ve hücre olduğundan değişkenlerin varyans değerleri birbirinden çok farklı olacağından bu durum elde edeceğimiz çıktıların güvenilirliğini azaltıyor. Bunun önüne geçmeye çalışıyoruz.*

In [4]:
from sklearn.preprocessing import StandardScaler

In [5]:
scaler = StandardScaler()
scaler.fit(X_train)

StandardScaler()

In [6]:
X_train_scaled = scaler.transform(X_train)

In [7]:
X_test_scaled = scaler.transform(X_test)

In [8]:
from sklearn.neural_network import MLPRegressor

In [9]:
mlp_model = MLPRegressor(hidden_layer_sizes = (100,20)).fit(X_train_scaled, y_train)

In [10]:
#katman sayisi

In [11]:
mlp_model.n_layers_

4

In [12]:
#gizli katmandaki noron sayisi

In [13]:
mlp_model.hidden_layer_sizes

(100, 20)

## Tahmin

In [14]:
mlp_model.predict(X_train_scaled)[0:5]

array([ 68.06038453, 217.50365036, 162.34059119,  57.52472977,
        40.88815048])

In [15]:
y_pred = mlp_model.predict(X_test_scaled)

In [16]:
#rmse

In [17]:
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(535.495856315013)

## Model Tuning

In [18]:
mlp_params = {'alpha' : [0.1, 0.01, 0.02, 0.005],
              'hidden_layer_sizes' : [(20, 20), (100, 50, 150), (300, 200, 150)],
              'activation' : ['relu', 'logistic']}

In [19]:
mlp_cv_model = GridSearchCV(mlp_model, mlp_params, cv = 10)

In [20]:
mlp_cv_model.fit(X_train_scaled, y_train)

GridSearchCV(cv=10, estimator=MLPRegressor(hidden_layer_sizes=(100, 20)),
             param_grid={'activation': ['relu', 'logistic'],
                         'alpha': [0.1, 0.01, 0.02, 0.005],
                         'hidden_layer_sizes': [(20, 20), (100, 50, 150),
                                                (300, 200, 150)]})

In [21]:
mlp_cv_model.best_params_

{'activation': 'relu', 'alpha': 0.005, 'hidden_layer_sizes': (100, 50, 150)}

*Burada binlerce kombinasyon denenebilir. Şu an modelin kendisine seçtiği parametre değerlerinin en iyi en optimum sonuçları vereceğinin bir garantisini veremeyiz.*

In [22]:
mlp_tuned = MLPRegressor(alpha = 0.1, hidden_layer_sizes = (100, 50, 150))

In [23]:
mlp_tuned.fit(X_train_scaled, y_train)

MLPRegressor(alpha=0.1, hidden_layer_sizes=(100, 50, 150))

In [24]:
#tune edilmis modelin rmse degeri

In [25]:
y_pred = mlp_tuned.predict(X_test_scaled)

In [26]:
np.sqrt(mean_squared_error(y_test, y_pred))

np.float64(357.714295659097)